In [21]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Working with Strings & Dates")
    .master("local[*]")
    .getOrCreate()
)

spark

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [22]:
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [23]:
emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

In [24]:
emp.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [25]:
emp.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [61]:
from pyspark.sql.functions import when, col, expr, regexp_replace, to_date, current_date, current_timestamp, to_timestamp

In [27]:
emp_gende = emp.withColumn("new_gender", when(col("gender")=='Male', "M").when(col("gender")=="Female", "F").otherwise(None))
emp_gender.show()


+-----------+-------------+-------------+---+------+------+----------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|
+-----------+-------------+-------------+---+------+------+----------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|         M|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|         F|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|         M|
|        010|   

In [31]:
emp_genders = emp.withColumn("New_column", expr("case when gender ='Male' then 'M' when gender = 'Female' then 'F' else null end  "))

emp_genders.show()

+-----------+-------------+-------------+---+------+------+----------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|New_column|
+-----------+-------------+-------------+---+------+------+----------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|         M|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|         F|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|         M|
|        010|   

**Replace in Strings**
**select employee_id, name, replace(name, 'J', 'Z') as new_name, age, salary, gender, new_gender, hire_date from emp_gender_fixed**

In [72]:
emp_name = emp_gender.withColumn("new_name", regexp_replace(col("name"), "J","Z"))
emp_name.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|    Bob Brown|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack Chan|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|    Zill Wong|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|         M|Zames Zohnson|
|        008|          102|     Kate Kim

In [76]:
# Convert Date
emp_date_fixed = emp_name.withColumn("hire_date_stamp", expr("try_to_timestamp(hire_date, 'yyyy-MM-dd')"))

emp_date_fixed.printSchema()

emp_date_fixed.show()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)
 |-- new_gender: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- hire_date_stamp: timestamp (nullable = true)

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+-------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|    hire_date_stamp|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+-------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|2015-01-01 00:00:00|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|2016-02-15 00:00:00|
|     

In [78]:
emp_dated = emp_date_fixed.withColumn("current_date", current_date()).withColumn("current_timestamp", current_timestamp())

emp_dated.limit(10).show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+-------------------+------------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|    hire_date_stamp|current_date|   current_timestamp|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+-------------------+------------+--------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|2015-01-01 00:00:00|  2025-11-25|2025-11-25 22:29:...|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|2016-02-15 00:00:00|  2025-11-25|2025-11-25 22:29:...|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|    Bob Brown|2014-05-01 00:00:00|  2025-11-25|2025-11-25 22:29:...|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|2017-

In [84]:
emp_1 = emp_name.na.drop()

emp_1.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|    Bob Brown|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack Chan|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|    Zill Wong|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|         M|Zames Zohnson|
|        008|          102|     Kate Kim

In [85]:
from pyspark.sql.functions import coalesce, lit

emp_null_df = emp_dated.withColumn("new_gender", coalesce(col("new_gender"), lit("0")))

emp_null_df.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+-------------------+------------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|    hire_date_stamp|current_date|   current_timestamp|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+-------------------+------------+--------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|2015-01-01 00:00:00|  2025-11-25|2025-11-25 22:36:...|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|2016-02-15 00:00:00|  2025-11-25|2025-11-25 22:36:...|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|    Bob Brown|2014-05-01 00:00:00|  2025-11-25|2025-11-25 22:36:...|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|2017-

In [91]:
# Drop old columns and Fix new column names
emp_final = emp_null_df.drop("name", "gender").withColumnRenamed("new_name", "name").withColumnRenamed("new_gender", "gender")

emp_final.show()


+-----------+-------------+---+------+----------+------+-------------+-------------------+------------+--------------------+
|employee_id|department_id|age|salary| hire_date|gender|         name|    hire_date_stamp|current_date|   current_timestamp|
+-----------+-------------+---+------+----------+------+-------------+-------------------+------------+--------------------+
|        001|          101| 30| 50000|2015-01-01|     M|     Zohn Doe|2015-01-01 00:00:00|  2025-11-25|2025-11-25 22:43:...|
|        002|          101| 25| 45000|2016-02-15|     F|   Zane Smith|2016-02-15 00:00:00|  2025-11-25|2025-11-25 22:43:...|
|        003|          102| 35| 55000|2014-05-01|     M|    Bob Brown|2014-05-01 00:00:00|  2025-11-25|2025-11-25 22:43:...|
|        004|          102| 28| 48000|2017-09-30|     F|    Alice Lee|2017-09-30 00:00:00|  2025-11-25|2025-11-25 22:43:...|
|        005|          103| 40| 60000|2013-04-01|     M|    Zack Chan|2013-04-01 00:00:00|  2025-11-25|2025-11-25 22:43:...|


In [94]:
#write date as csv

emp_final.write.format("csv").mode("overwrite").save("Datasets/emp.csv")

In [99]:
#now let convert date into a string and extract date information

from pyspark.sql.functions import date_format

emp_fixed = emp_final.withColumn("date_year", date_format(col("current_timestamp"), "z"))

emp_fixed.select("date_year").show()

+---------+
|date_year|
+---------+
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
|      CAT|
+---------+

